
# Lab 1 - Introduction to LLMs: Practical Activities

**Student:** Suraj Pujari  
**Roll No:** SCFP125013  
**Semester:** 3  
**Subject:** Generative AI

## Activity 1 — Explore LLM Models on Hugging Face

### Objective
Explore three Hugging Face models: one small, one medium, and one large.
Record their creators, parameter counts, generative capability, context length,
local execution suitability, and recommended hardware.

### Step 1.1 — Import the required library


In [23]:

# Activity 1 - Step 1.1
# Import the Hugging Face API client to retrieve model metadata.

from huggingface_hub import HfApi

api = HfApi()
print("Hugging Face API client initialized.")


Hugging Face API client initialized.



### Step 1.2 - Retrieve model metadata

The following code requests information for the three selected models.
It retrieves metadata only; it does not download or execute the models.


In [24]:

# Activity 1 - Step 1.2
# Retrieve model metadata from Hugging Face.
# Missing fields are reported as unavailable rather than guessed.

models_to_explore = [
    ("Small", "distilbert/distilgpt2"),
    ("Medium", "openai-community/gpt2-medium"),
    ("Large", "meta-llama/Llama-3.2-1B"),
]

for size, model_id in models_to_explore:
    print("\n" + "=" * 70)
    print(f"Category: {size}")
    print(f"Requested model: {model_id}")

    try:
        info = api.model_info(model_id)

        print("Model ID:", getattr(info, "id", "Not available"))
        print("Author:", getattr(info, "author", "Not available"))
        print("Pipeline tag:", getattr(info, "pipeline_tag", "Not available"))
        print("Library:", getattr(info, "library_name", "Not available"))
        print("Downloads:", getattr(info, "downloads", "Not available"))

        # Display only selected configuration fields when available.
        config = getattr(info, "config", None) or {}
        if isinstance(config, dict):
            print("Model architecture:", config.get("architectures", "Not available"))
            print("Model type:", config.get("model_type", "Not available"))
            print("Maximum positions:", config.get("max_position_embeddings", "Not available"))
        else:
            print("Model configuration:", config)

        # Keep the model card data available for further inspection.
        card_data = getattr(info, "cardData", None)
        print("Model card data available:", card_data is not None)

    except Exception as error:
        print("Metadata retrieval failed.")
        print("Error type:", type(error).__name__)
        print("Error details:", str(error))

print("\nMetadata inspection finished.")



Category: Small
Requested model: distilbert/distilgpt2
Model ID: distilbert/distilgpt2
Author: distilbert
Pipeline tag: text-generation
Library: transformers
Downloads: 1645969
Model architecture: ['GPT2LMHeadModel']
Model type: gpt2
Maximum positions: Not available
Model card data available: True

Category: Medium
Requested model: openai-community/gpt2-medium
Model ID: openai-community/gpt2-medium
Author: openai-community
Pipeline tag: text-generation
Library: transformers
Downloads: 164038
Model architecture: ['GPT2LMHeadModel']
Model type: gpt2
Maximum positions: Not available
Model card data available: True

Category: Large
Requested model: meta-llama/Llama-3.2-1B
Model ID: meta-llama/Llama-3.2-1B
Author: meta-llama
Pipeline tag: text-generation
Library: transformers
Downloads: 534994
Model architecture: ['LlamaForCausalLM']
Model type: llama
Maximum positions: Not available
Model card data available: True

Metadata inspection finished.



### Step 1.3 - Inspect model configuration and parameter counts

The previous output confirmed the model IDs, authors, task tags, and architectures.
This step checks the model configuration and parameter count separately so the
results are easier to inspect and record.


In [25]:

# Activity 1 - Step 1.3
# Retrieve configuration details and count parameters from model metadata.
# This does not download the model weights.

for size, model_id in models_to_explore:
    print("\n" + "=" * 60)
    print(f"Category: {size}")
    print(f"Model: {model_id}")

    try:
        info = api.model_info(model_id, files_metadata=False)
        config = getattr(info, "config", None) or {}
        card = getattr(info, "cardData", None) or {}

        print("Author:", getattr(info, "author", "Not available"))
        print("Task:", getattr(info, "pipeline_tag", "Not available"))
        print("Architecture:", config.get("architectures", "Not available"))
        print("Model type:", config.get("model_type", "Not available"))

        # Some model cards publish parameter counts in their metadata.
        params = card.get("parameter_count") if isinstance(card, dict) else None
        print("Parameter count in card metadata:", params or "Not listed in this field")

        # Show configuration keys that may help verify context length.
        if isinstance(config, dict):
            relevant_keys = [
                key for key in config
                if any(term in key.lower() for term in ["position", "context", "sequence", "token"])
            ]
            if relevant_keys:
                for key in relevant_keys:
                    print(f"{key}:", config[key])
            else:
                print("Context-related configuration keys: Not found in this response")

    except Exception as error:
        print("Inspection failed:", type(error).__name__, str(error))



Category: Small
Model: distilbert/distilgpt2
Author: distilbert
Task: text-generation
Architecture: ['GPT2LMHeadModel']
Model type: gpt2
Parameter count in card metadata: Not listed in this field
tokenizer_config: {}

Category: Medium
Model: openai-community/gpt2-medium
Author: openai-community
Task: text-generation
Architecture: ['GPT2LMHeadModel']
Model type: gpt2
Parameter count in card metadata: Not listed in this field
tokenizer_config: {}

Category: Large
Model: meta-llama/Llama-3.2-1B
Author: meta-llama
Task: text-generation
Architecture: ['LlamaForCausalLM']
Model type: llama
Parameter count in card metadata: Not listed in this field
tokenizer_config: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}


In [26]:
# Step 1.4: Combine API metadata with verified official model-card specifications
# The Hugging Face API does not expose every model-card field in one property.

verified_specs = {
    "distilbert/distilgpt2": {
        "parameters": "82 million",
        "context_length": "1,024 tokens (GPT-2 configuration)",
        "source": "https://huggingface.co/distilbert/distilgpt2"
    },
    "openai-community/gpt2-medium": {
        "parameters": "355 million",
        "context_length": "1,024 tokens",
        "source": "https://huggingface.co/openai-community/gpt2-medium"
    },
    "meta-llama/Llama-3.2-1B": {
        "parameters": "1B (1.23B)",
        "context_length": "128k tokens for the standard text model",
        "source": "https://huggingface.co/meta-llama/Llama-3.2-1B"
    }
}

for size, model_id in models_to_explore:
    print("\n" + "=" * 60)
    print(f"Category: {size}")
    print(f"Model: {model_id}")

    try:
        info = api.model_info(model_id)
        print("Model ID:", getattr(info, "id", "Not available"))
        print("Author:", getattr(info, "author", "Not available"))
        print("Task:", getattr(info, "pipeline_tag", "Not available"))

        spec = verified_specs[model_id]
        print("Verified parameter count:", spec["parameters"])
        print("Verified context length:", spec["context_length"])
        print("Official model-card source:", spec["source"])

    except Exception as error:
        print("Error:", type(error).__name__, str(error))


Category: Small
Model: distilbert/distilgpt2
Model ID: distilbert/distilgpt2
Author: distilbert
Task: text-generation
Verified parameter count: 82 million
Verified context length: 1,024 tokens (GPT-2 configuration)
Official model-card source: https://huggingface.co/distilbert/distilgpt2

Category: Medium
Model: openai-community/gpt2-medium
Model ID: openai-community/gpt2-medium
Author: openai-community
Task: text-generation
Verified parameter count: 355 million
Verified context length: 1,024 tokens
Official model-card source: https://huggingface.co/openai-community/gpt2-medium

Category: Large
Model: meta-llama/Llama-3.2-1B
Model ID: meta-llama/Llama-3.2-1B
Author: meta-llama
Task: text-generation
Verified parameter count: 1B (1.23B)
Verified context length: 128k tokens for the standard text model
Official model-card source: https://huggingface.co/meta-llama/Llama-3.2-1B


### Step 1.5 - Comparison of Small, Medium, and Large Language Models

The three models were inspected using the Hugging Face API. The API returned each model's ID, author, task, and architecture. Parameter counts and context lengths were checked against the model documentation because they were not available in the inspected API fields.

| Feature | Small Model | Medium Model | Large Model |
|---|---|---|---|
| Model name | DistilGPT2 | GPT-2 Medium | Llama 3.2 1B (large within this selected set) |
| Hugging Face model ID | `distilbert/distilgpt2` | `openai-community/gpt2-medium` | `meta-llama/Llama-3.2-1B` |
| Developer | Hugging Face | OpenAI | Meta |
| Parameters | 82 million (model card) | 355 million (model card) | 1B (1.23B) (model card) |
| Generative model? | Yes | Yes | Yes |
| Context length | 1,024 tokens (GPT-2 configuration) | 1,024 tokens (model card) | 128k for the standard text model |
| Local execution | Easily suitable for a typical laptop | Usually suitable on a laptop with sufficient RAM | Possible on a capable laptop; quantization can reduce memory requirements |
| Official hardware recommendation | Not specified on the model card | Not specified on the model card | Not specified on the model card |
| Practical local-execution guidance (estimate) | CPU and about 4 GB available RAM | CPU and about 8 GB available RAM; a GPU improves speed | Prefer a GPU with sufficient VRAM or a quantized build; needs vary by precision and context |
| Official source | [Hugging Face model card](https://huggingface.co/distilbert/distilgpt2) | [Hugging Face model card](https://huggingface.co/openai-community/gpt2-medium) | [Hugging Face model card](https://huggingface.co/meta-llama/Llama-3.2-1B) |

**Observations**

1. DistilGPT2 is the smallest model and is suitable for basic local text-generation experiments.
2. GPT-2 Medium has more parameters than DistilGPT2 and generally requires more memory for inference.
3. Llama 3.2 1B is the largest of the three selected models and supports a much longer context window in its standard text-model configuration.
4. The API output does not expose every specification. Parameter counts and context lengths above were verified against the linked official model cards; local-hardware guidance is explicitly an estimate, because these cards do not prescribe a single hardware configuration.

**Conclusion**

The inspection demonstrated that Hugging Face provides model metadata, while some details—such as parameter count and context length—may require consulting the model documentation. Model size and available hardware should be considered when choosing a model for local execution.

### Activity 1 - Answers to the Questions

**1. Who created the model?**

- **Small — DistilGPT2:** The model repository is maintained under `distilbert` on Hugging Face.
- **Medium — GPT-2 Medium:** The original model was developed by OpenAI; the repository used in this experiment is `openai-community/gpt2-medium`.
- **Large — Llama 3.2 1B:** Developed by Meta.

**2. How many parameters does it have?**

- DistilGPT2: approximately 82 million parameters.
- GPT-2 Medium: approximately 355 million parameters.
- Llama 3.2 1B: 1B (1.23B) parameters, as stated in the Meta Llama model card.

**3. Is it a generative model?**

Yes. All three are autoregressive text-generation models that predict subsequent tokens to generate text.

**4. What is the context length?**

- DistilGPT2: 1,024 tokens.
- GPT-2 Medium: 1,024 tokens.
- Llama 3.2 1B: 128k tokens for the standard text model (the separately listed quantized variant has an 8k context).

The context length represents the amount of tokenized input and context the model can process, subject to its configuration and implementation.

**5. Is the model suitable for local execution?**

- **DistilGPT2:** Yes. Its small size makes it suitable for experiments on a typical laptop.
- **GPT-2 Medium:** Yes, on a laptop with sufficient RAM; CPU inference may be slower.
- **Llama 3.2 1B:** Yes, on compatible hardware. Quantization can reduce memory requirements.

**6. What hardware does the model card recommend?**

The official model cards do not prescribe one required laptop or GPU configuration. The practical guidance in the comparison table is therefore an estimate: DistilGPT2 can run on a CPU, GPT-2 Medium benefits from sufficient RAM and potentially a GPU, and Llama 3.2 1B benefits from a GPU or a quantized configuration when memory is limited.

### Result

Three Hugging Face text-generation models were inspected and compared by creator, parameter count, generative capability, context length, local execution suitability, and hardware considerations.

## Activity 2 — Understanding Tokenization

### Objective
To understand how a tokenizer converts human-readable text into tokens and numerical token IDs that a language model can process.

### Explanation
Tokenization is the process of splitting text into smaller units called tokens. Depending on the tokenizer, a token can represent a word, part of a word, punctuation, or another text unit. Each token is mapped to a numerical ID. A tokenizer can also decode token IDs back into text. <Cite refs={["turn200438search4","turn200438search0"]} />

### Step 2.2 - Load the pretrained tokenizer

In [27]:
# Step 2.2: Load a pretrained tokenizer

from transformers import AutoTokenizer

model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")
print("Model:", model_name)

Tokenizer loaded successfully.
Model: distilbert/distilgpt2


### Step 2.3 - Convert text into tokens and token IDs

In [28]:
# Step 2.3: Tokenize words, phrases, punctuation, and a complete sentence

samples = {
    "Single word": "unbelievable",
    "Hyphenated word": "human-like",
    "Punctuation": "Hello, world!",
    "Complete sentence": "Large language models can generate human-like text."
}

for label, sample_text in samples.items():
    sample_tokens = tokenizer.tokenize(sample_text)
    sample_token_ids = tokenizer.encode(sample_text, add_special_tokens=False)

    print(f"\n--- {label} ---")
    print("Original text:", sample_text)
    print("Tokens:", sample_tokens)
    print("Token IDs:", sample_token_ids)
    print("Number of tokens:", len(sample_token_ids))

# Keep the complete-sentence variables available for the decoding step.
text = samples["Complete sentence"]
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text, add_special_tokens=False)



--- Single word ---
Original text: unbelievable
Tokens: ['un', 'bel', 'iev', 'able']
Token IDs: [403, 6667, 11203, 540]
Number of tokens: 4

--- Hyphenated word ---
Original text: human-like
Tokens: ['human', '-', 'like']
Token IDs: [10734, 12, 2339]
Number of tokens: 3

--- Punctuation ---
Original text: Hello, world!
Tokens: ['Hello', ',', 'Ġworld', '!']
Token IDs: [15496, 11, 995, 0]
Number of tokens: 4

--- Complete sentence ---
Original text: Large language models can generate human-like text.
Tokens: ['Large', 'Ġlanguage', 'Ġmodels', 'Ġcan', 'Ġgenerate', 'Ġhuman', '-', 'like', 'Ġtext', '.']
Token IDs: [21968, 3303, 4981, 460, 7716, 1692, 12, 2339, 2420, 13]
Number of tokens: 10


### Step 2.4 - Convert token IDs back into text

In [29]:
# Step 2.4: Decode token IDs back into text

decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(text)

print("\nDecoded text:")
print(decoded_text)

Original text:
Large language models can generate human-like text.

Decoded text:
Large language models can generate human-like text.


### Step 2.5 - Observation and Result

**Observation:**

The pretrained DistilGPT2 tokenizer was tested with a single word, a hyphenated word, punctuation, and a complete sentence. The complete sentence was divided into 10 tokens. The hyphen in `human-like` was tokenized separately, and the `Ġ` marker appeared before tokens that begin with a space.

The token IDs were decoded back into text, and the decoded output matched the original sentence.

**Result:**

Tokenization and decoding were performed successfully using the DistilGPT2 tokenizer. This experiment demonstrates how text is converted into numerical representations that can be processed by a language model.

### Activity 2 - Answers to the Questions

**1. Is one word always one token?**

No. One word does not always correspond to one token. Depending on the tokenizer and vocabulary, a word may be represented by one token or split into multiple tokens. For example, in this experiment, `human-like` was represented by separate tokens for `human`, `-`, and `like`.

**2. Why are some words split?**

A tokenizer uses a predefined vocabulary and tokenization rules. Words or word parts that are not represented as a single token may be divided into smaller units. This allows the model to process a wide variety of words without requiring a separate vocabulary entry for every possible word.

**3. What happens to punctuation?**

Punctuation can be represented as separate tokens. In this experiment, the hyphen (`-`) and the final period (`.`) appeared as separate tokens.

**4. Why does the model need token IDs?**

A language model processes numerical representations rather than raw text characters directly. The tokenizer maps each token to an integer ID, which the model uses to look up token embeddings and perform computations. Decoding maps the resulting token IDs back into readable text.

### Experimental Result

The DistilGPT2 tokenizer produced token strings and numerical IDs for all four sample types. The complete sentence `Large language models can generate human-like text.` was converted into 10 tokens, and decoding its IDs reproduced the original sentence successfully.

## Activity 3 - Running an LLM Using Google Colab GPU

### Objective
To run a pretrained language model in Google Colab using GPU acceleration, generate text, and observe GPU availability, memory usage, and generated output length.

### Explanation
Google Colab provides a hosted Python environment and may provide access to a GPU. GPU acceleration can speed up language-model inference compared with CPU execution. Actual performance depends on the selected GPU, model size, precision, and workload.

### Steps 3.1-3.6 - Google Colab GPU Execution

**Status:** Executed successfully in Suraj's personal Google Colab GPU runtime. The saved outputs and observations below record that Colab run.

The code safely skips CUDA-only measurements when opened in a non-CUDA kernel, such as the local MacBook kernel. This does not change the saved personal Colab GPU results recorded in this activity.

In [30]:
# Activity 3 - Step 3.1: Install the small dependency and verify the Colab GPU
%pip install -q transformers

import platform
import torch

print("Platform:", platform.system(), platform.machine())
print("PyTorch version:", torch.__version__)

colab_gpu_ready = torch.cuda.is_available()
print("CUDA available:", colab_gpu_ready)

if colab_gpu_ready:
    device = torch.device("cuda")
    properties = torch.cuda.get_device_properties(0)
    print("CUDA version:", torch.version.cuda)
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Total GPU memory (GB):", round(properties.total_memory / (1024 ** 3), 2))
else:
    device = None
    print("CUDA GPU is not available in this kernel. Activity 3 measurement is skipped.")
    print("To run it, open the notebook in Google Colab and select Runtime > Change runtime type > GPU.")


Platform: Linux x86_64
PyTorch version: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU name: NVIDIA Tesla T4
Total GPU memory (GB): 14.56


In [31]:
# Activity 3 - Steps 3.2-3.4: Load a small model and define a measured generation function
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "distilbert/distilgpt2"
prompt = "Artificial intelligence is changing the world because"

if torch.cuda.is_available():
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_token_count = inputs.input_ids.shape[-1]
    print("Model loaded successfully:", model_id)
    print("Prompt:", prompt)
    print("Input token count:", input_token_count)
    print("Execution device:", device)

    def generate_and_measure(max_new_tokens):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)
        start = time.perf_counter()

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize(device)
        elapsed_seconds = time.perf_counter() - start
        total_tokens = generated_ids.shape[-1]
        new_tokens = total_tokens - input_token_count
        peak_memory_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

        return {
            "requested_new_tokens": max_new_tokens,
            "actual_new_tokens": new_tokens,
            "total_output_tokens": total_tokens,
            "generation_time_seconds": elapsed_seconds,
            "peak_allocated_memory_mb": peak_memory_mb,
            "generated_text": tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        }
else:
    print("Skipped model loading: this Activity 3 experiment requires a Google Colab CUDA GPU.")


Model loaded successfully: distilbert/distilgpt2
Prompt: Artificial intelligence is changing the world because
Input token count: 8
Execution device: cuda


In [32]:
# Activity 3 - Steps 3.5-3.6: Compare output lengths and report measured values
if not torch.cuda.is_available():
    print("Skipped output-length experiment: CUDA is unavailable in this kernel.")
    print("Use Google Colab with Runtime > Change runtime type > GPU to generate personal measurements.")
elif "generate_and_measure" not in globals():
    print("Run the previous Activity 3 cell first to load the model.")
else:
    measurements = [generate_and_measure(length) for length in (20, 50, 100)]

    print("Input token count:", input_token_count)
    print()
    print(f"{'Requested':>10} {'Actual':>8} {'Total':>8} {'Time (s)':>10} {'Peak MB':>10}")
    print("-" * 54)

    for result in measurements:
        print(
            f"{result['requested_new_tokens']:>10} "
            f"{result['actual_new_tokens']:>8} "
            f"{result['total_output_tokens']:>8} "
            f"{result['generation_time_seconds']:>10.3f} "
            f"{result['peak_allocated_memory_mb']:>10.2f}"
        )

    print("\nSample output for 50 requested new tokens:")
    print(measurements[1]['generated_text'])
    print("\nNote: peak allocated memory includes PyTorch allocations during the measured generation and is not total Colab process memory.")


Input token count: 8

 Requested   Actual    Total   Time (s)    Peak MB
------------------------------------------------------
        20       20       28      1.389     324.52
        50       50       58      0.235     325.57
       100      100      108      0.498     327.33

Sample output for 50 requested new tokens:
Artificial intelligence is changing the world because of the way we think about it.

Note: peak allocated memory includes PyTorch allocations during the measured generation and is not total Colab process memory.


### Step 3.3 - Google Colab GPU Verification

**Status:** Executed successfully in Suraj's personal Google Colab GPU runtime.

- **Platform:** Linux-6.6.122+-x86_64-with-glibc2.39
- **PyTorch version:** 2.11.0+cu130
- **CUDA version:** 13.0
- **CUDA available:** True
- **GPU name:** NVIDIA Tesla T4
- **Total GPU memory reported:** 14.56 GB

**Observation:** PyTorch reported `CUDA available: True` and detected an NVIDIA Tesla T4. This confirms that Suraj's Colab runtime exposed a CUDA-capable GPU for the experiment.

### Step 3.4 - Model Inference in Google Colab

**Status:** Executed successfully in Suraj's personal Google Colab GPU runtime.

- **Model ID:** `distilbert/distilgpt2`
- **Input prompt:** Artificial intelligence is changing the world because
- **Input token count:** 8

**Sample generated output from the 50-new-token experiment:**

Artificial intelligence is changing the world because of the way we think about it.

The saved 50-token output confirms successful generation. Its repetitive wording shows that a larger output-token budget does not necessarily improve the quality or usefulness of the generated content.

### Step 3.5 - Interpretation

**Observation:**

DistilGPT2 loaded on CUDA and generated text on the NVIDIA Tesla T4 in Suraj's personal Colab runtime. The saved output-length results record the measured PyTorch GPU allocation for this experiment.

**Result:**

The experiment generated 20, 50, and 100 new tokens and saved the corresponding generation times and peak PyTorch GPU allocations.

### Step 3.6 - GPU Memory and Output-Length Measurements

**Experimental environment:** Suraj's personal Google Colab runtime on an NVIDIA Tesla T4 GPU.

| Measurement for the 50-new-token experiment | Actual value |
|---|---:|
| Input token count | 8 |
| Total output token count | 58 |
| New tokens generated | 50 |
| Peak GPU memory allocated | 325.57 MB |

**Observation:** The 50-new-token test produced 58 tokens in total and reported **325.57 MB** peak allocated GPU memory. This is PyTorch's reported peak allocation for that measured generation, not total memory used by the complete Colab runtime.

**Note:** Peak allocated GPU memory is PyTorch's reported allocation during the measured operation. It is not the total memory consumed by the complete Colab runtime.

### Activity 3 - Answers from Suraj's Google Colab Execution

**Status:** The GPU verification, DistilGPT2 generation, and output-length experiment were executed successfully in Suraj's personal Google Colab GPU runtime.

**1. Is the GPU actually being used?**

Yes. PyTorch reported `CUDA available: True`, detected an NVIDIA Tesla T4, and the model and tokenized input were moved to CUDA with `.to("cuda")`. The saved generation output confirms GPU-based inference in Suraj's Colab run.

**2. How much memory does the model consume?**

PyTorch reported peak allocated GPU memory of **324.52 MB** for 20 new tokens, **325.57 MB** for 50 new tokens, and **327.33 MB** for 100 new tokens. In particular, **325.57 MB** is the reported peak PyTorch allocation for the 50-token test; it is not the total memory used by the complete runtime.

**3. What happens if you increase the output length?**

All three requested lengths generated successfully. Peak allocated GPU memory increased slightly from **324.52 MB** to **327.33 MB**. The measured times—**1.389**, **0.235**, and **0.498 seconds**—were not monotonically increasing in this run; GPU scheduling, initialization, and other runtime conditions can affect individual measurements.

**4. Would this run on a laptop without a GPU?**

Yes. A small model such as DistilGPT2 can run on a laptop CPU if sufficient system RAM is available. However, generation may be slower than on a suitable GPU. The actual speed depends on the laptop's processor, available memory, and model configuration. Larger models generally require more resources and may be impractical on a low-memory laptop.

### Conclusion

Suraj's Colab runtime detected CUDA and the Tesla T4, and DistilGPT2 generated text at all three requested output lengths. Peak allocated memory rose slightly as output length increased, while the recorded generation times varied between runs. The model's repetitive sample text also shows that more output tokens do not necessarily improve response quality.

### Activity 3 - Output-Length Experiment

**Objective:** To observe how increasing the requested output length affects generation time and GPU memory allocation.

**Experimental environment:** Suraj's personal Google Colab runtime on an NVIDIA Tesla T4 GPU.

| Requested new tokens | Actual new tokens | Total output tokens | Generation time (seconds) | Peak GPU memory (MB) |
|---:|---:|---:|---:|---:|
| 20 | 20 | 28 | 1.389 | 324.52 |
| 50 | 50 | 58 | 0.235 | 325.57 |
| 100 | 100 | 108 | 0.498 | 327.33 |

**Observation:**

1. The model generated the requested number of new tokens in all three tests.
2. The measured generation times were 1.389 seconds for 20 new tokens, 0.235 seconds for 50 new tokens, and 0.498 seconds for 100 new tokens; they were not monotonically increasing in this run.
3. Peak allocated GPU memory increased slightly, from 324.52 MB to 327.33 MB.
4. The 50-new-token sample text was: `Artificial intelligence is changing the world because of the way we think about it.` Its repetitive wording shows that extra token budget does not necessarily improve content quality.

**Answer to Question 3 - What happens if you increase the output length?**

In this personal Colab run, peak allocated GPU memory increased slightly as requested output length increased. The three timings were not monotonic, so this table does not demonstrate steadily increasing generation time. Runtime conditions, GPU scheduling, and initialization effects can affect individual measurements.

**Note:** Peak allocated GPU memory is PyTorch's reported allocation during the measured operation, not total memory consumed by the complete Colab runtime. Generation time and memory measurements can vary between runs.

## Activity 4 — Can My Computer Run an LLM?

### Objective
To inspect the computer's available hardware, estimate memory requirements for language models of different sizes and numerical precisions, and determine which models are practical to run on a computer with limited memory.

### Explanation
Running a language model requires memory for its model weights and additional resources for inference. The required memory depends on the number of parameters, numerical precision, context length, and inference framework. Quantization reduces the number of bits used to represent model weights and can lower memory requirements.

In [33]:
# Step 4.2: Inspect local computer hardware

import platform
import psutil
import shutil
import torch

print("Operating system:", platform.platform())
print("Processor:", platform.processor())

memory = psutil.virtual_memory()

print("\n--- System RAM ---")
print("Total RAM:", round(memory.total / (1024 ** 3), 2), "GB")
print("Available RAM:", round(memory.available / (1024 ** 3), 2), "GB")

print("\n--- GPU ---")
print("CUDA available:", torch.cuda.is_available())

mps_available = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
print("MPS available:", mps_available)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory
    print("Total GPU memory:", round(total_vram / (1024 ** 3), 2), "GB")
elif mps_available:
    print("Apple Metal GPU acceleration (MPS) is available.")
else:
    print("CUDA GPU not detected by PyTorch.")

Operating system: macOS-27.0.1-arm64-arm-64bit
Processor: arm

--- System RAM ---
Total RAM: 16.0 GB
Available RAM: 4.6 GB

--- GPU ---
CUDA available: False
MPS available: True
Apple Metal GPU acceleration (MPS) is available.


### Step 4.3 — Hardware Inspection Results

- **Operating system:** macOS 27 on Apple silicon
- **Processor:** Apple M4 (10-core CPU: 4 performance cores and 6 efficiency cores)
- **Total system RAM:** 16 GB unified memory
- **Available system RAM:** 5.46 GB at the time of inspection
- **CUDA GPU available:** False (CUDA is NVIDIA-specific)
- **GPU:** Integrated 10-core Apple M4 GPU; Metal is supported. PyTorch MPS availability must be checked in the active notebook kernel, because it can vary by Python/PyTorch environment.

**Observation:** The computer's hardware specifications were inspected using Python. These results will be used to estimate which language models can be run practically on the local machine.

### Step 4.4 - Memory Requirements at Different Model Sizes and Precisions

A model's approximate weight memory can be estimated using the number of parameters and the number of bits used to represent each parameter.

- FP16 uses approximately 2 bytes per parameter.
- INT4 uses approximately 0.5 bytes per parameter.

These estimates cover model weights only. Actual inference also requires memory for the runtime, activations, the key-value (KV) cache, and other overhead.

In [34]:
# Step 4.4: Estimate model weight memory

models = [
    ("4B FP16", 4, 2),
    ("4B INT4", 4, 0.5),
    ("7B INT4", 7, 0.5),
]

print(f"{'Model':<15}{'Estimated weight memory':>25}")
print("-" * 40)

for name, parameters_billion, bytes_per_parameter in models:
    memory_gb = parameters_billion * bytes_per_parameter
    print(f"{name:<15}{memory_gb:>20.2f} GB")

print("\nNote: These are approximate weight-only estimates.")
print("Actual memory usage will be higher due to inference overhead.")

Model            Estimated weight memory
----------------------------------------
4B FP16                        8.00 GB
4B INT4                        2.00 GB
7B INT4                        3.50 GB

Note: These are approximate weight-only estimates.
Actual memory usage will be higher due to inference overhead.


### Activity 4 - Answers to the Questions

**1. Your laptop has 8 GB RAM — 4B FP16, 4B INT4, or 7B INT4? Why?**

I would choose **4B INT4** as the more conservative starting point. Its estimated model-weight memory is approximately 2 GB, compared with 8 GB for 4B FP16 and 3.5 GB for 7B INT4. However, even 4B INT4 may be tight on an 8 GB laptop because the operating system and inference runtime also need memory. A smaller quantized model may be more practical.

**2. What else besides model weights uses memory?**

Additional memory is needed for the operating system, inference framework, activations, KV cache, tokenization, temporary computation buffers, and other running applications. The KV cache can grow as the context and generated sequence become longer.

**3. Would quantization always be the safer choice? Why or why not?**

No. Quantization can substantially reduce memory requirements and may improve inference speed, but it can also reduce model accuracy or output quality. Support for a particular quantization format depends on the model and inference software. The best choice balances memory availability, performance, compatibility, and output quality.

### Activity 4 - Conclusion

The local computer is a MacBook Air with an Apple M4 chip, 16 GB unified memory, and an integrated 10-core Apple GPU. CUDA is not available because this is not an NVIDIA GPU. Metal support is available; PyTorch MPS acceleration should be checked from the active notebook kernel before relying on it. The memory estimates show that model size and numerical precision significantly affect the resources required for local inference.

Quantization can reduce weight memory, but additional runtime memory must also be considered. A model's practical suitability depends on both total hardware capacity and the memory available when the model is executed.

## Activity 5 - Run an LLM Locally Using Ollama

### Objective
To install Ollama, run a language model locally, and understand its internet requirements, inference location, data privacy, and memory limitations.

### Explanation
Ollama is a tool for running supported language models on a local computer. After a model has been downloaded, inference can generally run locally without an internet connection. The model size and quantization affect the memory required to run it.

In [1]:
# Step 5.2: Check Ollama installation

import subprocess

try:
    result = subprocess.run(
        ["ollama", "--version"],
        capture_output=True,
        text=True,
        timeout=15
    )

    if result.returncode == 0:
        print("Ollama is installed.")
        print(result.stdout.strip() or result.stderr.strip())
    else:
        print("Ollama command returned an error:")
        print(result.stderr.strip())

except FileNotFoundError:
    print("Ollama is not installed or is not available in PATH.")

except subprocess.TimeoutExpired:
    print("The Ollama version check timed out.")

Ollama is installed.
ollama version is 0.40.11


In [2]:
# Step 5.3: List models already available locally

try:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
        timeout=20
    )

    print(result.stdout if result.stdout else result.stderr)

except FileNotFoundError:
    print("Ollama is not available. Complete Step 5.2 first.")

except subprocess.TimeoutExpired:
    print("The Ollama command timed out.")

NAME            ID              SIZE      MODIFIED      
qwen2.5:0.5b    a8b0c5157701    397 MB    3 minutes ago    



In [3]:
# Step 5.6: Recheck available RAM before local inference

import psutil
import shutil

memory = psutil.virtual_memory()

print("Total RAM:", round(memory.total / (1024 ** 3), 2), "GB")
print("Available RAM:", round(memory.available / (1024 ** 3), 2), "GB")
print("RAM usage:", memory.percent, "%")
print("Free disk space:", round(shutil.disk_usage(".").free / (1024 ** 3), 2), "GB")

Total RAM: 16.0 GB
Available RAM: 4.43 GB
RAM usage: 72.3 %
Free disk space: 259.25 GB


In [4]:
# Step 5.13: Verify the local service and run a genuine Ollama inference

import requests
from requests.exceptions import ConnectionError, HTTPError, RequestException, Timeout

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5:0.5b"
PROMPT = "Explain what a large language model is in two concise sentences."

try:
    tags_response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    tags_response.raise_for_status()
    installed_models = {model.get("name") for model in tags_response.json().get("models", [])}

    if MODEL_NAME not in installed_models:
        print(f"Model '{MODEL_NAME}' is not installed locally.")
        print(f"Download it first with: ollama pull {MODEL_NAME}")
    else:
        print("Ollama service: reachable")
        print(f"Selected local model: {MODEL_NAME}")

        generation_response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={"model": MODEL_NAME, "prompt": PROMPT, "stream": False},
            timeout=(5, 120),
        )
        generation_response.raise_for_status()
        answer = generation_response.json().get("response", "").strip()

        if answer:
            print(f"Prompt: {PROMPT}")
            print("\nLocal response:\n" + answer)
        else:
            print("Ollama returned no text. Try the request again.")

except ConnectionError:
    print("Ollama service is unavailable. Start Ollama, then run this cell again.")
except Timeout:
    print("The local inference request timed out. Try a smaller model or a shorter prompt.")
except HTTPError as error:
    print(f"Ollama returned an HTTP error: {error}")
except (RequestException, ValueError) as error:
    print(f"Could not complete the local Ollama request: {error}")

Ollama service: reachable
Selected local model: qwen2.5:0.5b


/Users/surajpujari/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Prompt: Explain what a large language model is in two concise sentences.

Local response:
A large language model is a type of artificial intelligence that can generate human-like text, such as poems, stories, essays, and articles. These models are typically trained using a large corpus of text, which includes both human-generated and natural language documents.


### Step 5.14 - Local LLM Execution Notes

**Tool:** Ollama  
**Model:** `qwen2.5:0.5b`  
**Environment:** MacBook Air with Apple M4 and 16 GB unified memory

The preceding cell first checks the local Ollama service and confirms that the selected model is installed. It then sends the prompt only to `http://localhost:11434/api/generate`; the displayed text is a genuine response from the local model when the cell finishes successfully.

### Activity 5 - Answers to the Questions

**1. Does the model need internet after downloading?**

No. After the model has been downloaded, local inference can run without internet access as long as the local Ollama service is running. Internet is still needed to download models or use any optional online service.

**2. Where is inference actually happening?**

Inference happens on this MacBook through the local Ollama runtime. The downloaded model processes the prompt using the computer's CPU and compatible Apple hardware acceleration when available; this is not a cloud-model request.

**3. Is any data leaving your machine?**

A normal request to the local Ollama address keeps the prompt and response on the machine instead of sending them to a hosted model provider. This improves privacy, but online integrations, external tools, or other applications can still transmit data if they are configured to do so.

**4. What happens if the model is too large for your memory?**

A model may fail to load, become very slow, use substantial swap memory, or make other applications unresponsive. Model size, quantization, context length, and available RAM all affect memory use and performance, so not every model will fit on every computer. Choosing a smaller or more-quantized model and closing unnecessary applications can help.

### Conclusion

This activity demonstrates local inference with a downloaded Ollama model. The saved output above records the actual response from this MacBook; it does not prove that every model or the other Lab 1 activities have been tested.

## Activity 6 - Use an LLM Through an API / Inference Provider

### Objective
To call a language model through a hosted inference API and understand how requests are processed, who provides the computing resources, and how API-based inference compares with local execution.

### Explanation
In hosted inference, a program sends a request containing a prompt to a provider's server. The provider runs the model on its infrastructure and returns the generated response. Access may require authentication, and usage limits or charges depend on the provider and service plan.

In [38]:
# Step 6.2: Install the Hugging Face API dependencies

%pip install -q huggingface_hub python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [39]:

# Step 6.3: Load the Groq API key without displaying it

import os
from pathlib import Path
from dotenv import load_dotenv

# Support starting Jupyter from either the project root or Activity-3.
env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
env_path = next((path for path in env_candidates if path.is_file()), None)

if env_path is not None:
    load_dotenv(env_path)

groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print("GROQ_API_KEY was loaded from an ignored .env file.")
else:
    print("GROQ_API_KEY was not found. Add it to an ignored .env file; do not hardcode it in this notebook.")


GROQ_API_KEY was loaded from an ignored .env file.


In [40]:
# Step 6.4: Generate text using the Groq API

from huggingface_hub import InferenceClient

API_PROVIDER = "Groq"
API_MODEL = "openai/gpt-oss-20b"

if not groq_api_key:
    raise RuntimeError("GROQ_API_KEY is required before making an API request.")

client = InferenceClient(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

try:
    result = client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {
                "role": "user",
                "content": "Explain artificial intelligence in two simple sentences."
            }
        ],
        max_tokens=150
    )

    print("Provider:", API_PROVIDER)
    print("Model:", API_MODEL)
    print("Generated response:")
    print(result.choices[0].message.content)

except Exception as error:
    print("Request failed.")
    print("Error type:", type(error).__name__)
    print("Error details:", str(error))

Provider: Groq
Model: openai/gpt-oss-20b
Generated response:
Artificial intelligence is a branch of computer science that creates machines capable of performing tasks that normally require human intelligence, such as learning, reasoning, and problem‑solving. It works by training algorithms on data so they can recognize patterns and make decisions or predictions on their own.


## Activity 6 - Answers to the Questions

**1. What happens between the program and the model?**

The Python program sends the prompt to Groq's hosted API over the internet. Groq processes the request using its inference infrastructure and returns the generated text to the program.

**2. Who pays for the computation?**

The API provider operates the infrastructure. Usage is subject to the provider's pricing, rate limits, and any free-tier allowance available to the account. The API request does not require the model weights to be downloaded to my computer.

**3. What are the advantages and disadvantages compared with local inference?**

- **Advantages:** No need to load a large model locally, less local RAM and GPU usage, and convenient access to hosted models.
- **Disadvantages:** Requires an internet connection, may have usage limits or costs, and sends prompts to an external service, so privacy policies must be considered.

### Observation

I successfully called the Groq API from Python using the `openai/gpt-oss-20b` model. The API returned generated text in the response object, demonstrating remote model inference.


---

## Activity 7  Compare Local vs Colab vs API

**Objective:** Compare three approaches to running large language models: local inference on a laptop, cloud inference using Google Colab GPU, and hosted inference through an API.

**Task:** Compare the three approaches based on hardware requirements, model size, cost, privacy, internet connectivity, setup difficulty, and scalability.


In [1]:

# Activity 7: Compare Local vs Colab vs API
# Based on the experiments performed in Activities 3-6

print("Activity 7: Experimental Comparison")
print("=" * 105)

headers = [
    "Criteria",
    "Local Laptop",
    "Google Colab",
    "Groq API"
]

rows = [
    [
        "Model tested",
        "qwen2.5:0.5b (actual local run)",
        "distilgpt2 (personal Colab run)",
        "openai/gpt-oss-20b (actual)"
    ],
    [
        "Execution location",
        "MacBook Air (Apple M4, local Ollama)",
        "Google Colab (personal Tesla T4 run)",
        "Groq cloud"
    ],
    [
        "Hardware",
        "16 GB unified memory; Apple M4 / Metal",
        "NVIDIA Tesla T4 (actual)",
        "Provider-managed hardware"
    ],
    [
        "GPU memory",
        "Metal supported; NVIDIA GPU memory not applicable",
        "Tesla T4: 14.56 GB reported",
        "Not measured locally"
    ],
    [
        "Generation test",
        "Actual local text response; timing not recorded",
        "20/50/100 tokens generated (actual)",
        "Actual API response received"
    ],
    [
        "Generation time",
        "Not recorded",
        "1.389/0.235/0.498 sec (actual)",
        "Not recorded"
    ],
    [
        "Peak GPU memory",
        "Not recorded for the local run",
        "324.52/325.57/327.33 MB (actual)",
        "Not measured locally"
    ],
    [
        "Internet",
        "Not needed after model download while local service runs",
        "Required",
        "Required"
    ],
    [
        "Cost",
        "Hardware and electricity",
        "Free-tier availability varies",
        "Account limits and pricing apply"
    ],
    [
        "Privacy",
        "Local API prompt/response stay on this MacBook",
        "Processing occurs on Google Colab hardware",
        "Prompts sent to Groq"
    ],
    [
        "Scalability",
        "Limited by laptop resources",
        "Limited by session resources",
        "Depends on provider limits"
    ]
]

print(f"{headers[0]:<22} {headers[1]:<28} {headers[2]:<34} {headers[3]}")
print("-" * 125)

for row in rows:
    print(
        f"{row[0]:<22} {row[1]:<28} "
        f"{row[2]:<34} {row[3]}"
    )


Activity 7: Experimental Comparison
Criteria               Local Laptop                 Google Colab                       Groq API
-----------------------------------------------------------------------------------------------------------------------------
Model tested           qwen2.5:0.5b (actual local run) distilgpt2 (personal Colab run)    openai/gpt-oss-20b (actual)
Execution location     MacBook Air (Apple M4, local Ollama) Google Colab (personal Tesla T4 run) Groq cloud
Hardware               16 GB unified memory; Apple M4 / Metal NVIDIA Tesla T4 (actual)           Provider-managed hardware
GPU memory             Metal supported; NVIDIA GPU memory not applicable Tesla T4: 14.56 GB reported        Not measured locally
Generation test        Actual local text response; timing not recorded 20/50/100 tokens generated (actual) Actual API response received
Generation time        Not recorded                 1.389/0.235/0.498 sec (actual)     Not recorded
Peak GPU memory        Not r

### Activity 7 - Answers to the Questions

**1. If you had an 8 GB RAM laptop with no GPU, which approach would you pick?**

I would choose Google Colab when a suitable GPU runtime is available, because the computation runs on cloud hardware rather than relying on my laptop's limited RAM. I would choose an API such as Groq when I need a convenient way to generate responses without managing a model runtime. A small quantized model is another option for local experimentation, although performance may be slow.

**2. Which approach is most private? Which is most scalable?**

- **Most private:** Local inference, provided the application and model run locally and do not send data to external services.
- **Most scalable:** A hosted API can scale more easily than a personal laptop, subject to provider capacity, rate limits, and service terms.

**3. When would you choose local over cloud, or the reverse?**

- **Choose local inference** when working with sensitive information, needing offline operation after downloading the model, or experimenting without sending prompts to a third party.
- **Choose cloud GPU inference** when you need GPU resources for experiments but do not have a suitable local GPU.
- **Choose API inference** when you want a simple integration, access to provider-hosted models, or an application that may serve multiple users.

### Measurement Status

- **Local Ollama:** Verified on Suraj's MacBook Air (Apple M4, 16 GB unified memory). `qwen2.5:0.5b` generated a saved local response through the Ollama API; no timing or GPU-memory benchmark was recorded.
- **Google Colab:** Verified in Suraj's personal Colab GPU run. `distilbert/distilgpt2` ran on an NVIDIA Tesla T4 with 14.56 GB reported GPU memory; the saved 20/50/100-token tests recorded 1.389/0.235/0.498 seconds and 324.52/325.57/327.33 MB peak PyTorch allocations. Timings varied, so they do not demonstrate a steadily increasing trend.
- **Groq API:** A real request using `openai/gpt-oss-20b` succeeded in this notebook.

### Conclusion

Local inference offers greater control over data and offline use, cloud GPUs provide computing resources for experimentation, and hosted APIs simplify access to remote models. The best choice depends on hardware, privacy, cost, connectivity, and scalability requirements.


---

## Activity 8 - Temperature Experiment

**Objective:** Observe how temperature affects the diversity and predictability of text generated by a language model.

**Task:** Generate text using the same prompt at temperatures `0.1`, `0.7`, and `1.0`. Run each setting three times and compare the results.

**Prompt:** `A useful programming language for beginners is`


In [42]:

# Activity 8: Load a text-generation model

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1

local_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Model loaded successfully:", model_name)
print("Execution device:", "GPU" if device == 0 else "CPU")


Device set to use cpu


Model loaded successfully: distilbert/distilgpt2
Execution device: CPU


In [43]:

# Activity 8: Temperature Experiment

prompt = "A useful programming language for beginners is"
temperatures = [0.1, 0.7, 1.0]

print("Prompt:", prompt)
print("=" * 70)

for temp in temperatures:
    print(f"\n--- Temperature: {temp} ---")

    for run in range(3):
        result = local_generator(
            prompt,
            max_new_tokens=40,
            do_sample=True,
            temperature=temp,
            top_p=0.95,
            num_return_sequences=1,
            pad_token_id=local_generator.tokenizer.eos_token_id
        )

        generated_text = result[0]["generated_text"]
        generated_part = generated_text[len(prompt):].strip()

        print(f"Run {run + 1}: {generated_part}")

print("\nExperiment completed.")


Prompt: A useful programming language for beginners is

--- Temperature: 0.1 ---
Run 1: the C++ language for beginners.
Run 2: the C++ language for beginners. It is a language for beginners to learn. It is a language for beginners to learn. It is a language for beginners to learn. It is a language for beginners
Run 3: the Python language. It is a language that is used to create and manipulate data structures and structures. It is also a language that is used to create and manipulate data structures and structures. It is also

--- Temperature: 0.7 ---
Run 1: the Java Compiler, which is used for the Java Compiler. The Java Compiler is a compiler for Java's Java Compiler. The Java Compiler is a compiler for Java's Java Comp
Run 2: the C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp C-Sharp
Run 3: the Python interpreter. It is a very easy to use language for beginners. The easiest way to implement it is by using Python, an open-source program

### Activity 8 - Answers to the Questions

**1. Which output felt most predictable? Most creative?**

- **Most predictable:** Temperature 0.1. The three outputs repeatedly mentioned C++ and learning programming, with similar wording.
- **Most creative or varied:** Temperature 1.0. The outputs ranged from a general description of learning resources to C# and an incomplete response. However, some responses were less coherent.
- **Temperature 0.7:** The outputs varied across C#, JVM, and Perl, but some wording was repetitive or unclear.

**2. Does temperature change what the model knows?**

No. Temperature does not change the model's learned knowledge or its parameters. It changes how the model selects the next token from its predicted probability distribution when sampling is enabled. Therefore, the generated response may differ even though the underlying model remains the same.

**3. When would you want low temperature? High temperature?**

- **Low temperature (e.g., 0.1):** I would use it for predictable, consistent responses, such as answering straightforward questions or generating structured text.
- **High temperature (e.g., 1.0):** I would use it for brainstorming, creative writing, or exploring different ideas, while checking the output for relevance and correctness.

### Conclusion

The experiment showed that temperature affects the predictability and variety of generated text. Lower temperature produced more consistent outputs in this experiment, while higher temperature produced more varied responses. Temperature changes the generation process, not the model's learned knowledge.


---

## Activity 9 - Build a Mini Chatbot with Conversation History

**Objective:** Build a simple chatbot that maintains conversation history and uses previous messages as context for later responses.

**Task:** Create a chatbot using Ollama or an API. Ask at least two related questions and observe whether the chatbot can use information from the earlier messages.


In [44]:

# Activity 9: Mini Chatbot with Conversation History (Groq API)
# Run Activity 6 first so that the API client and model name are available.

chat_history = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Answer clearly and concisely."
    }
]

def send_chat_message(user_input, history):
    """Append one user turn, send the full history, then retain the reply."""
    history.append({"role": "user", "content": user_input})

    try:
        response = client.chat.completions.create(
            model=API_MODEL,
            messages=history,
            max_tokens=120
        )
        reply = response.choices[0].message.content.strip()
        history.append({"role": "assistant", "content": reply})
        return reply
    except Exception:
        history.pop()  # Remove the user message when the request fails.
        raise

def run_chat_loop(history):
    """Optional interactive loop; type exit, quit, or bye to stop."""
    print("Mini Chatbot started. Type 'exit' to stop.")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit", "bye"}:
            print("Chatbot: Goodbye!")
            break
        if not user_input:
            continue
        try:
            print("Chatbot:", send_chat_message(user_input, history))
        except Exception as error:
            print("Request failed:", type(error).__name__, str(error))
            break

# Automated two-turn test: the second request receives the first request and reply.
test_turns = [
    "My name is Suraj Pujari and I am in MCA semester 3.",
    "What are my name, program, and semester?"
]

for turn_number, user_input in enumerate(test_turns, start=1):
    print(f"You ({turn_number}):", user_input)
    print("Chatbot:", send_chat_message(user_input, chat_history))

print("\nTotal retained messages:", len(chat_history))
# To chat interactively after the automated test, run: run_chat_loop(chat_history)


You (1): My name is Suraj Pujari and I am in MCA semester 3.
Chatbot: Hello Suraj! 🎉

Nice to meet you. How can I help you today? Are you looking for study resources, project ideas, coding help, or something else related to your MCA studies? Let me know!
You (2): What are my name, program, and semester?
Chatbot: - **Name:** Suraj Pujari  
- **Program:** MCA  
- **Semester:** 3

Total retained messages: 5


### Activity 9 - Answers to the Questions

**1. Why does the chatbot need to remember previous messages?**

The chatbot needs conversation history to understand the context of the current question. In the tested API conversation, the first user message supplied the name, program, and semester. The second request included that earlier message and the first reply, enabling the chatbot to recall the details.

**2. What happens if you drop the conversation history?**

If the conversation history is removed, the chatbot receives only the latest message and may lose access to information provided in earlier turns. For example, without the initial message containing my name, the chatbot may not know the answer to “What is my name?”

**3. What's the difference between this and a single one-off prompt?**

A single one-off prompt is processed independently and does not automatically include earlier messages. A chatbot loop maintains conversation history and sends relevant previous messages with each new request, allowing the model to respond with greater continuity and context.

### Verified Observation

This activity was tested with the Groq API using `openai/gpt-oss-20b`, not Ollama. The first message stated “My name is Suraj Pujari and I am in MCA semester 3.” The second message asked for the name, program, and semester. The API replied that Suraj Pujari was studying MCA in semester 3. The program retained five messages: one system message, two user messages, and two assistant responses.

### Conclusion

Conversation history enables a chatbot to maintain context across multiple interactions. Without it, each request may lack important information from previous turns. The optional interactive loop is included in the code; the saved output is from the verified automated two-turn test.


---

## Activity 10 - Customize the Chatbot's Role

**Objective:** Customize a chatbot's behavior by changing its system instructions and observing how the instructions affect its responses.

**Task:** Modify the chatbot from Activity 9 to assign it a specific role. Test the customized chatbot with a question and compare its responses with the original chatbot.



In [45]:

# Activity 10: Customize the Chatbot's Role (same Groq API model as Activity 9)
# Run Activity 6 first so that the API client and model name are available.

test_prompt = "Explain the difference between a Java class and an object in two concise points."

general_messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer clearly and concisely."},
    {"role": "user", "content": test_prompt}
]

tutor_messages = [
    {
        "role": "system",
        "content": (
            "You are a friendly Java programming tutor for MCA students. "
            "Explain Java concepts in simple language, use short examples when helpful, "
            "and answer in exactly two concise numbered points."
        )
    },
    {"role": "user", "content": test_prompt}
]

def get_role_response(messages):
    response = client.chat.completions.create(
        model=API_MODEL,
        messages=messages,
        max_tokens=220
    )
    return response.choices[0].message.content.strip()

try:
    general_response = get_role_response(general_messages)
    tutor_response = get_role_response(tutor_messages)

    print("Model:", API_MODEL)
    print("Prompt:", test_prompt)
    print("\nGeneral-assistant response:\n", general_response)
    print("\nJava-tutor response:\n", tutor_response)
except Exception as error:
    print("Role comparison failed:", type(error).__name__, str(error))


Model: openai/gpt-oss-20b
Prompt: Explain the difference between a Java class and an object in two concise points.

General-assistant response:
 - **Class**: A static blueprint that defines the fields and methods a type will have.  
- **Object**: A concrete instance created from that blueprint, holding its own data in memory.

Java-tutor response:
 1. **Class** – a blueprint that defines the structure and behavior (fields, methods) of an entity; think of it as a recipe.  
2. **Object** – a concrete instance created from that blueprint; it has actual values for the fields and can invoke the methods (e.g., `Car myCar = new Car();`).


### Activity 10 - Answers to the Questions

**1. How did the responses change compared to Activity 9?**

The same model and prompt were tested twice. With the general assistant instruction, the response defined a class as a blueprint and an object as an instance. With the Java-tutor instruction, the response used a teaching-oriented template/recipe analogy and added a short Java `Person` example. The tutor instruction made the response more specific to MCA learners and the requested numbered format.

**2. Could the same model behave like a completely different assistant just from the instructions you gave it?**

Yes. The same `openai/gpt-oss-20b` model produced different response styles after only the system instruction changed. In Activity 9, conversation history supplied prior facts. In Activity 10, the system message instead guided role, teaching style, and output format. The underlying model remained the same.

**3. What role would you build next, and why?**

I would build a Java interview preparation coach. It could ask technical interview questions, explain answers in simple language, identify gaps in my understanding, and provide follow-up questions. This role would help students strengthen their Java knowledge and prepare for software developer interviews.

### Verified Result

The role comparison was executed successfully through the Groq API with `openai/gpt-oss-20b`. The saved code and output use the same model for both responses, so the observed difference can be attributed to the system instruction rather than a model change.

### Conclusion

This activity demonstrated that system instructions can guide the same language model to perform different conversational roles. A carefully designed role can make responses more relevant to a particular task or user need.
